# Langchain Agents Integration

* So far our application has following two options,
    * A simple  RAG option - we retrieve the embeddings from the vector db using user entered text and fixed number of documents and augment the LLM reponse with this context. 
    * Optimized query otpion (chains) - we use LLM to create the query using user text and then use the LLM generated query to retrieve the embeddings from the database. 
* These options produce great results for simple queries, but for advanced metadata specific queries to fails to retrieve accurate response. 
* For that we'll need agentic option that will allow an LLM to reason and use different tools to retrieve and respond to user queries. 

## Import Libraries

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain.agents import load_tools, initialize_agent
from langchain.agents import AgentType
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError


import os
import pandas as pd
import numpy as np
import chromadb

## Initialize DB

In [2]:
embedding_function = OpenAIEmbeddings()
vector_db = Chroma(persist_directory="../db/dunder_bot", embedding_function=embedding_function)

In [3]:
## lets verify the db connection
vector_db._collection.count()

29228

## Creating Tools    

## Tools

In [15]:
data_dir = Path("..","data","raw/")
metadata_path = Path(data_dir,"metadata.json")

In [19]:
## read metadata json
import json


with open(metadata_path,"r") as fp:
    metadata = json.load(fp=fp)


In [24]:
season_metadata = [data for data in metadata if data["season"] == 1]
season_metadata[0]

{'season': 1,
 'total_episodes': 6,
 'episodes_ratings': [7.5, 8.3, 7.8, 8.1, 8.4, 7.8],
 'episodes_dates': ['2005-03-24',
  '2005-03-29',
  '2005-04-05',
  '2005-04-12',
  '2005-04-19',
  '2005-04-26'],
 'directors': ['Ken Kwapis',
  'Ken Whittingham',
  'Bryan Gordon',
  'Greg Daniels',
  'Amy Heckerling'],
 'writers': ['Ricky Gervais',
  'Stephen Merchant',
  'Greg Daniels',
  'B. J. Novak',
  'Paul Lieberstein',
  'Michael Schur',
  'Greg Daniels',
  'Mindy Kaling'],
 'characters': ['Michael',
  'Jim',
  'Pam',
  'Dwight',
  'Jan',
  'Michel',
  'Todd Packer',
  'Phyllis',
  'Stanley',
  'Oscar',
  'Angela',
  'Kevin',
  'Ryan',
  'Man',
  'Roy',
  'Documentary Crew Member',
  'Mr. Brown',
  'Toby',
  'Kelly',
  'Meredith',
  'Travel Agent',
  'Man on Phone',
  'Everybody',
  'Lonny',
  'Darryl',
  'Teammates',
  'Michael and Dwight',
  'Warehouse worker',
  'Madge',
  'Worker',
  'Packer',
  'Warehouse Worker',
  'Katy']}

In [36]:
from langchain.agents import tool
from langgraph.prebuilt import create_react_agent


@tool
def get_total_seasons_of_the_office(text: str) -> int:
    """
    Returns the total number of seasons in the TV show The Office. Use only when the user’s question is clearly about the show. \
    The input should always be an empty string \
    and this function will always return a integer with total number of seasons. 
    This function cannot be used for any other season info
    Args:
        text (str): empty string

    Returns:
        int: integer representing total number of season in "The Office"
    """
    return 9


@tool
def get_season_info_of_the_office(season: str) -> dict:
    """Retrieves detailed metadata for a specific season of the TV show *The Office*.

    Use this tool when a user asks for comprehensive information about a single season.
    It is ideal for questions about directors, writers, main characters, episode ratings, 
    or the total number of episodes within that season.

    Args:
        season (str): The season number, which must be provided as a string (e.g., "1", "2").

    Returns:
        dict: A dictionary containing the season's metadata with the following keys:
            - 'season' (int): The season number.
            - 'total_episodes' (int): The total count of episodes in the season.
            - 'episodes_ratings' (list[float]): A list of IMDb ratings for each episode.
            - 'episodes_dates' (list[str]): A list of original air dates for each episode.
            - 'directors' (list[str]): A list of unique directors for the season.
            - 'writers' (list[str]): A list of unique writers for the season.
            - 'characters' (list[str]): A list of main characters featured in the season.
        
        Returns an empty dictionary ({}) if the requested season is not found.
    """
    # get season metadata
    season_metadata = [data for data in metadata if data["season"] == season]

    if len(season_metadata) > 0:
        return season_metadata[0]

    return {}

## Initializing Agent

In [40]:
# list of tools for the agent to use
custom_tools = [get_total_seasons_of_the_office, get_season_info_of_the_office]

# initialize llm model
llm_model = "gpt-3.5-turbo"
llm = ChatOpenAI(temperature=0, model=llm_model)

# set system prompt
system_prompt = """
You are DunderBot, the official AI assistant for the Dunder Mifflin Paper Company's Scranton branch. Your entire world revolves around the events, characters, and dialogue from the American version of the TV show *The Office*.
Your sole purpose is to answer questions about *The Office*. If a user asks about any other topic (like other TV shows, real-world events, or science), you MUST politely decline and remind them of your specialized role.
To answer questions, you have access to a set of specialized tools. Always prioritize using these tools to find factual information. Your final answers must be based *only* on the information retrieved by the tools. This prevents you from making things up or hallucinating.
Adopt a witty, fun, and quirky tone appropriate for the show, but ensure your answers are always accurate and true to the show's canon.
"""


# initialize agent
# agent = initialize_agent(
#     llm=llm,
#     agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
#     tools=custom_tools,
#     verbose=True,
#     agent_kwargs={
#         "system_prompt": system_prompt
#     },
#     handle_parsing_errors=True
# )

agent = create_react_agent(
    model=llm_model,
    tools=custom_tools,
    prompt=system_prompt,
)

## Running Agent

In [41]:
user_content = "how many episodes are there in season 3 of the office?"

try:
    response = agent.invoke({"messages": [{"role": "user", "content": user_content}]})
except GraphRecursionError:
    print("Agent stopped due to max iterations.")

In [42]:
response

{'messages': [HumanMessage(content='how many episodes are there in season 3 of the office?', additional_kwargs={}, response_metadata={}, id='cc607583-bcf8-444d-9168-1f700c278372'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_bqQJ8bo6UriQi2WZZk1KhsP8', 'function': {'arguments': '{"season":"3"}', 'name': 'get_season_info_of_the_office'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 597, 'total_tokens': 615, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Bi2l0lk17TDFv4kay2OEGonLzuCsP', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-65c402ae-e373-493d-a803-c4cf854e5409-0', tool_calls=[{'name': 'get_season_info_of_the_office', 'args': {'season': '3

In [14]:

try:
    response = agent.invoke("how many seasons are there in the country India?"),
        # {"recursion_limit": recursion_limit},
except GraphRecursionError:
    print("Agent stopped due to max iterations.")



> Entering new AgentExecutor chain...
Thought: This question is not about the TV show "The Office," so I cannot use the tool to get the total number of seasons. I will need to provide a response based on general knowledge.

Final Answer: The question is not clear as to what is meant by "seasons" in the context of the country India. If you are referring to weather seasons, India typically experiences three main seasons: summer, monsoon, and winter. If you are referring to TV show seasons, then the question would not apply to the country India.

> Finished chain.


In [9]:
response

({'input': 'how many seasons are there in the country India?',
  'output': 'I cannot determine the number of seasons in the country India as it does not relate to the TV show "The Office."'},)